In [2]:
#Imports
import numpy as np
import pandas as pd
import joblib
import time

from tensorflow.keras.models import load_model
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
#Loading saved models and encoders

# Paths (adjust only if folder changes)
NCF_MODEL_PATH = r"D:\MINI PROJECT\Checkpoints\ncfmodel.keras"
USER_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\user_encoder.pkl"
MOVIE_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\movie_encoder.pkl"
EMBEDDINGS_PATH = r"D:\MINI PROJECT\Checkpoints\movie_embeddings.npy"

MOVIES_PATH = r"D:\MINI PROJECT\DATASET\movies_final.csv"
RATINGS_PATH = r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv"

# Load NCF model
ncf_model = load_model(NCF_MODEL_PATH)

# Load encoders
user_encoder = joblib.load(USER_ENCODER_PATH)
movie_encoder = joblib.load(MOVIE_ENCODER_PATH)

print("Model & encoders loaded successfully")


Model & encoders loaded successfully


In [10]:
#Loading data
MOVIES_PATH = r"D:\MINI PROJECT\DATASET\movies_final.csv"
RATINGS_PATH = r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv"

movies_df = pd.read_csv(MOVIES_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)

ratings_df["timestamp"] = pd.to_numeric(
    ratings_df["timestamp"], errors="coerce"
)

ratings_df = ratings_df.dropna(subset=["timestamp"])


print("Movies:", movies_df.shape)
print("Ratings:", ratings_df.shape)


Movies: (23138, 15)
Ratings: (0, 4)


In [5]:
#Loading content embeddings
embeddings = np.load(EMBEDDINGS_PATH)

assert len(embeddings) == len(movies_df), "Embedding mismatch!"
print("Embeddings loaded:", embeddings.shape)


Embeddings loaded: (23138, 384)


In [1]:
#movie metadata
def get_movie_metadata(movie_id, movies_df):
    row = movies_df[movies_df.movieId == movie_id].iloc[0]
    return {
        "title": row["title"],
        "genres": row["genres"]
    }


In [6]:
#XAI explanation

def explain_content_similarity(
    user_id,
    recommended_movie_id,
    ratings_df,
    movies_df,
    embeddings,
    top_k=3
):
    movie_id_to_index = dict(zip(movies_df.movieId, movies_df.index))

    seen_movies = ratings_df[
        ratings_df.userId == user_id
    ]["movieId"].tolist()

    rec_idx = movie_id_to_index[recommended_movie_id]
    rec_vec = embeddings[rec_idx].reshape(1, -1)

    similarities = []

    for mid in seen_movies:
        if mid in movie_id_to_index:
            idx = movie_id_to_index[mid]
            sim = cosine_similarity(
                rec_vec,
                embeddings[idx].reshape(1, -1)
            )[0][0]
            similarities.append((mid, sim))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]


In [7]:
def explain_time_influence(user_id, ratings_df):
    now = time.time()

    user_ratings = ratings_df[
        ratings_df.userId == user_id
    ].copy()

    user_ratings["delta_days"] = (
        now - user_ratings["timestamp"]
    ) / (60 * 60 * 24)

    recent = user_ratings.sort_values("delta_days").iloc[0]

    return {
        "recent_movie_id": int(recent["movieId"]),
        "days_ago": round(recent["delta_days"], 2)
    }


In [8]:
def explain_recommendation(
    user_id,
    recommended_movie_id,
    ratings_df,
    movies_df,
    embeddings,
    alpha=0.7
):
    rec_meta = get_movie_metadata(
        recommended_movie_id,
        movies_df
    )

    similar_movies = explain_content_similarity(
        user_id,
        recommended_movie_id,
        ratings_df,
        movies_df,
        embeddings
    )

    time_info = explain_time_influence(
        user_id,
        ratings_df
    )

    explanation = {
        "recommended_movie": rec_meta["title"],
        "genres": rec_meta["genres"],
        "explanation": {
            "because_you_watched": [
                get_movie_metadata(mid, movies_df)["title"]
                for mid, _ in similar_movies
            ],
            "model_weights": {
                "collaborative (NCF)": alpha,
                "content_based": round(1 - alpha, 2)
            },
            "recent_activity": time_info
        }
    }

    return explanation


In [ ]:
USER_ID = 10

# Example: take a movie from hybrid recommendations
sample_movie_id = movies_df.iloc[100]["movieId"]

explanation = explain_recommendation(
    user_id=USER_ID,
    recommended_movie_id=sample_movie_id,
    ratings_df=ratings_df,
    movies_df=movies_df,
    embeddings=embeddings,
    alpha=0.7
)

explanation


{'recommended_movie': 'The Bridges of Madison County',
 'genres': 'drama, romance',
 'explanation': {'because_you_watched': ['The Bridge on the River Kwai',
   'The American President',
   'Leaving Las Vegas'],
  'model_weights': {'collaborative (NCF)': 0.7, 'content_based': 0.3},
  'recent_activity': {'recent_movie_id': 1, 'days_ago': np.float64(nan)}}}

GENERATING BETTER EXPLANATIONS


In [11]:
def generate_human_explanation(
    recommended_title,
    genres,
    similar_movies,
    alpha,
    recent_days=None
):
    explanation = []

    # Similarity explanation
    if similar_movies:
        explanation.append(
            f"This movie was recommended because you enjoyed "
            f"{', '.join(similar_movies[:3])}, which share similar themes and storytelling."
        )

    # Time-aware explanation
    if recent_days is not None and not np.isnan(recent_days):
        explanation.append(
            f"Your recent activity ({int(recent_days)} days ago) was given more importance "
            f"while generating this recommendation."
        )

    # Model balance explanation
    explanation.append(
        f"The recommendation balances collaborative filtering "
        f"({int(alpha*100)}%) and content-based similarity "
        f"({int((1-alpha)*100)}%)."
    )

    return " ".join(explanation)


In [12]:
def explain_recommendation(
    recommended_movie,
    genres,
    similar_movies,
    alpha,
    recent_days
):
    return {
        "movie": recommended_movie,
        "genres": genres,
        "explanation_text": generate_human_explanation(
            recommended_movie,
            genres,
            similar_movies,
            alpha,
            recent_days
        )
    }


In [5]:
!pip install datasets

  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0


You should consider upgrading via the 'D:\MINI PROJECT\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [6]:
from datasets import load_dataset

from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("avi-kai/Medical_Prescription_Handwritten_Words")

Resolving data files:   0%|          | 0/46 [00:00<?, ?it/s]

In [8]:
# Example for the 'train' split
print(ds['train'].shape)        # Output: (rows, cols)
print(ds['train'].num_rows)     # Output: total rows
print(ds['train'].num_columns)  # Output: total columns


(46, 1)
46
1
